## 1. Sequential Workflow

In [1]:
import wikipedia

def wikipedia_tool(query: str) -> str:
    """
    Searches Wikipedia for a given query and returns a summary of the top result.

    Args:
        query (str): The search term to look up on Wikipedia.
    """
    try:
        # The summary method automatically finds the best-matching page
        # and returns a summary of it.
        summary = wikipedia.summary(query)
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        # Handle cases where a query is ambiguous (e.g., "Java")
        return f"The query '{query}' is ambiguous. Please be more specific. Options: {e.options[:3]}"
    except wikipedia.exceptions.PageError:
        # Handle cases where the page does not exist
        return f"Sorry, I could not find a Wikipedia page for '{query}'."
    except Exception as e:
        return f"An unexpected error occurred while searching Wikipedia: {e}"

In [2]:
from google.adk.agents.llm_agent import LlmAgent

# Define the wikipedia agent

wikipedia_agent = LlmAgent(
    name='wikipedia_researcher',
    model='gemini-2.5-flash-lite',
    description='An expert at finding and summarizing information from Wikipedia.',
    instruction='You are a specialized agent and your only task is to extract the research TOPIC from the request and use the `wikipedia_tool` to find relevant information.',
    tools=[wikipedia_tool],
    output_key="wikipedia_notes"
)

In [3]:
def report_writer_tool(content: str, filename: str) -> str:
    """
    Writes the given content to a local file. Appends if the file already exists.

    Args:
        content (str): The text content to write to the file.
        filename (str): The name of the file to save the content in (e.g., 'report.txt').
    """
    try:
        # Use 'a' for append mode. This will create the file if it doesn't exist,
        # or add to the end of it if it does.
        with open(filename, 'a', encoding='utf-8') as f:
            f.write(content + "\n")
        return f"Successfully appended content to {filename}."
    except Exception as e:
        return f"An error occurred while writing to file: {e}"

In [4]:
from google.adk.agents.llm_agent import LlmAgent

# Define the writer agent

writer_agent = LlmAgent(
    name='report_writer',
    model='gemini-2.5-flash-lite',
    description='An expert at writing content to a file.',
    instruction=(
        "You are a specialized writing agent.\n"
        "Write a short report based on the user's request and the research notes below.\n\n"
        "Wikipedia notes:\n{wikipedia_notes}\n\n"
        "Then save the final report to text file using `report_writer_tool`."
        "The filename should be based on the research topic (e.g., black_holes_report.txt)..\n"
    ),    
    tools=[report_writer_tool]
)

Seq Workflow

In [5]:
from google.adk.agents.sequential_agent import SequentialAgent

# Define the sequential agent

root_agent = SequentialAgent(
    name="research_pipeline",
    description="Runs the research assistant steps in a fixed order.",
    sub_agents=[
        wikipedia_agent,
        writer_agent
    ],
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_9628\2490396848.py:5: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


## 2. Parallel WorkFLow

- Parallel research: We will use a ParallelAgent to execute the Wikipedia research and the arXiv research tasks concurrently. The system will start both jobs simultaneously and wait for both to complete.

- Sequential writing: Once the parallel research step is finished and both sets of notes are available, a SequentialAgent will proceed to the next stage, passing the combined results to our writer_agent to produce the final report.

Therefore a hybrid workflow

In [6]:
import wikipedia
import arxiv

# Defining the wikipedia tool

def wikipedia_tool(query: str) -> str:
    """
    Searches Wikipedia for a given query and returns a summary of the top result.

    Args:
        query (str): The search term to look up on Wikipedia.
    """
    try:
        # The summary method automatically finds the best-matching page
        # and returns a summary of it.
        summary = wikipedia.summary(query)
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        # Handle cases where a query is ambiguous (e.g., "Java")
        return f"The query '{query}' is ambiguous. Please be more specific. Options: {e.options[:3]}"
    except wikipedia.exceptions.PageError:
        # Handle cases where the page does not exist
        return f"Sorry, I could not find a Wikipedia page for '{query}'."
    except Exception as e:
        return f"An unexpected error occurred while searching Wikipedia: {e}"
        
# Defining the arxiv tool

def arxiv_tool(query: str) -> str:
    """
    Searches the arXiv repository for academic papers matching a query.

    Args:
        query (str): The topic to search for academic papers on.
    """
    try:
        # Create a client to interact with the arXiv API
        client = arxiv.Client()

        # Define the search parameters
        search = arxiv.Search(
            query=query,
            max_results=2,
            sort_by=arxiv.SortCriterion.Relevance
        )
        
        results = []
        # Use the client to execute the search and get the results
        for result in client.results(search):
            results.append(f"Title: {result.title}\nSummary: {result.summary}\nURL: {result.entry_id}")
            
        if not results:
            return f"No academic papers found on arXiv for the query '{query}'."
            
        return "\n---\n".join(results)
    except Exception as e:
        return f"An unexpected error occurred while searching arXiv: {e}"

In [7]:
from google.adk.agents.llm_agent import LlmAgent

# Define the wikipedia agent

wikipedia_agent = LlmAgent(
    name='wikipedia_researcher',
    model='gemini-2.5-flash-lite',
    description='An expert at finding and summarizing information from Wikipedia.',
    instruction='You are a specialized agent and your only task is to extract the core research TOPIC from the request and use the `wikipedia_tool` to find relevant information.',
    tools=[wikipedia_tool],
    output_key="wikipedia_notes"
)

# Define the arxiv agent

arxiv_agent = LlmAgent(
    name='arxiv_researcher',
    model='gemini-2.5-flash',
    description='An expert at finding and summarizing academic papers from the arXiv repository.',
    instruction='You are a specialized agent and your only task is to extract the core research TOPIC from the request and use the `arxiv_tool` to find relevant information.',
    tools=[arxiv_tool],
    output_key="arxiv_notes"
)

In [8]:
def report_writer_tool(content: str, filename: str) -> str:
    """
    Writes the given content to a local file. Appends if the file already exists.

    Args:
        content (str): The text content to write to the file.
        filename (str): The name of the file to save the content in (e.g., 'report.txt').
    """
    try:
        # Use 'a' for append mode. This will create the file if it doesn't exist,
        # or add to the end of it if it does.
        with open(filename, 'a', encoding='utf-8') as f:
            f.write(content + "\n")
        return f"Successfully appended content to {filename}."
    except Exception as e:
        return f"An error occurred while writing to file: {e}"

In [10]:
from google.adk.agents.llm_agent import LlmAgent

# Define the writer agent

writer_agent = LlmAgent(
    name='report_writer',
    model='gemini-2.5-flash-lite',
    description='An expert at writing content to a file.',
    instruction=(
        "You are a specialized writing agent.\n"
        "Write a short report based on the user's request and the research notes below.\n\n"
        "Wikipedia notes:\n{wikipedia_notes}\n\n"
        "arXiv notes:\n{arxiv_notes}\n\n"
        "Then save the final report to text file using `report_writer_tool`."
        "The filename should be based on the research topic (e.g., black_holes_report.txt)..\n"
    ),    
    tools=[report_writer_tool]
)

## Assembling the workflow

In [11]:
from google.adk.agents.parallel_agent import ParallelAgent

research_agent = ParallelAgent(
    name="parallel_research",
    description="Runs Wikipedia and arXiv research at the same time.",
    sub_agents=[
        wikipedia_agent,
        arxiv_agent
    ],
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_9628\2952226256.py:3: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  research_agent = ParallelAgent(


In [12]:
from google.adk.agents.sequential_agent import SequentialAgent

# Define the sequential agent

root_agent = SequentialAgent(
    name="research_pipeline",
    description="Runs parallel research first, then writes the report.",
    sub_agents=[
        research_agent, 
        writer_agent],
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_9628\3198102269.py:5: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


## 3. Loop Workflow

Loop termination condition :
- the tool formally tells the parent LoopAgent to stop running as soon as the current loop cycle finishes.

In [13]:
from google.adk.tools.tool_context import ToolContext

def exit_loop(tool_context: ToolContext) -> dict:
    """Stop the LoopAgent early."""
    tool_context.actions.escalate = True
    return {"status": "stopping loop"}

##### The iterative research agent

In [14]:
import arxiv

def arxiv_tool(query: str) -> str:
    """
    Searches the arXiv repository for academic papers matching a query.

    Args:
        query (str): The topic to search for academic papers on.
    """
    try:
        # Create a client to interact with the arXiv API
        client = arxiv.Client()

        # Define the search parameters
        search = arxiv.Search(
            query=query,
            max_results=2,
            sort_by=arxiv.SortCriterion.Relevance
        )
        
        results = []
        # Use the client to execute the search and get the results
        for result in client.results(search):
            results.append(f"Title: {result.title}\nSummary: {result.summary}\nURL: {result.entry_id}")
            
        if not results:
            return f"No academic papers found on arXiv for the query '{query}'."
            
        return "\n---\n".join(results)
    
    except Exception as e:
        return f"An unexpected error occurred while searching arXiv: {e}"

In [15]:
from google.adk.agents.llm_agent import LlmAgent

# Define the arxiv agent

arxiv_agent = LlmAgent(
    name="arxiv_researcher",
    model="gemini-2.5-flash-lite",
    description="Finds academic papers from arXiv.",
    instruction=(
        "You are a specialized agent whose ONLY job is to search arXiv.\n"
        "The user's message may be a report request. Extract the TOPIC.\n"
        "Call arxiv_tool(query=<topic>).\n"
        "If the tool returns papers, call exit_loop() to stop further iterations,\n"
        "then output the papers as your final response.\n"
        "If the tool returns 'No academic papers found', do NOT write a report.\n"
        "Just try again until max_iterations is reached."
    ),
    tools=[arxiv_tool, exit_loop],
    output_key="arxiv_notes",
)

The writer_agent will serve as the final step after the loop completes

In [16]:
from google.adk.agents.llm_agent import LlmAgent

def report_writer_tool(content: str, filename: str) -> str:
    """
    Writes the given content to a local file. Appends if the file already exists.

    Args:
        content (str): The text content to write to the file.
        filename (str): The name of the file to save the content in (e.g., 'report.txt').
    """
    try:
        # Use 'a' for append mode. This will create the file if it doesn't exist,
        # or add to the end of it if it does.
        with open(filename, 'a', encoding='utf-8') as f:
            f.write(content + "\n")
        return f"Successfully appended content to {filename}."
    except Exception as e:
        return f"An error occurred while writing to file: {e}"

# Define the writer agent

writer_agent = LlmAgent(
    name='report_writer',
    model='gemini-2.5-flash-lite',
    description='An expert at writing content to a file.',
    instruction=(
        "You are a specialized writing agent.\n"
        "Write a short report based on the user's request and the research notes below.\n\n"
        "arXiv notes:\n{arxiv_notes}\n\n"
        "Then save the final report to text file using `report_writer_tool`."
        "The filename should be based on the research topic (e.g., black_holes_report.txt)..\n"
    ),    
    tools=[report_writer_tool]
)

#### Creating the loop agent

In [17]:
from google.adk.agents.loop_agent import LoopAgent

# Define the loop agent

arxiv_loop = LoopAgent(
    name="arxiv_retry_loop",
    sub_agents=[arxiv_agent],
    max_iterations=2,
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_9628\2341160896.py:5: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  arxiv_loop = LoopAgent(


#### Final Pipeline

In [18]:
from google.adk.agents.sequential_agent import SequentialAgent

# Define the sequential agent

root_agent = SequentialAgent(
    name="pipeline",
    description="Runs arxiv loop first, then writes the report.",
    sub_agents=[
        arxiv_loop, 
        writer_agent],
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_9628\200843644.py:5: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(
